# Twin-XGB with New Features, Asymmetric Loss, and Time Series CV

Goals:
- Memory-safe cleaning and preprocessing (no sampling).
- Feature engineering for classification and regression targets.
- Time Series Cross Validation (expanding window).
- Two-stage XGBoost (zero classifier + regressor with asymmetric loss).

Notes:
- Recursive forecasting is intentionally deferred.
- Holidays are computed using country codes mapped from raw `Country`.


In [ ]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("mode.copy_on_write", True)

RANDOM_STATE = 42


## 1) Config and memory-safe IO

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("Could not locate project root containing 'src' directory.")

RAW_PATH = PROJECT_ROOT / "data/raw/online_retail.csv"

CHUNK_SIZE = 200_000
USECOLS = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price", "Country"]
DTYPES = {
    "Invoice": "string",
    "StockCode": "string",
    "Quantity": "float32",
    "Price": "float32",
    "Country": "string",
}

NON_PRODUCT_CODES = {
    "POST", "DOT", "C2", "M", "D", "ADJUST", "ADJUST2",
    "BANK CHARGES", "AMAZONFEE", "B", "S", "PADS",
    "TEST001", "TEST002", "GIFT_0001_10", "GIFT_0001_20",
    "GIFT_0001_30", "GIFT_0001_40", "GIFT_0001_50", "GIFT_0001_70",
    "GIFT_0001_80",
}


## 2) Chunked cleaning + daily aggregation (no sampling)

In [ ]:
def preprocess_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    df = chunk.rename(
        columns={
            "Invoice": "invoice",
            "StockCode": "stock_code",
            "Quantity": "quantity",
            "InvoiceDate": "invoice_date",
            "Price": "price",
            "Country": "country",
        }
    ).copy()

    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").astype("float32")
    df["price"] = pd.to_numeric(df["price"], errors="coerce").astype("float32")

    df["stock_code"] = df["stock_code"].astype("string").str.strip().str.upper()
    df["invoice"] = df["invoice"].astype("string").str.strip()
    df["country"] = df["country"].astype("string").str.strip()

    df = df.drop_duplicates()
    df = df[df["invoice_date"].notna()]
    df = df[df["stock_code"].notna() & (df["stock_code"] != "")]
    df = df[df["invoice"].notna() & (df["invoice"] != "")]

    df = df[~df["invoice"].str.startswith("C", na=False)]
    df = df[~df["stock_code"].isin(NON_PRODUCT_CODES)]

    df = df[(df["quantity"] > 0) & (df["price"] > 0)]

    df["date"] = df["invoice_date"].dt.normalize()
    df["revenue"] = df["quantity"] * df["price"]

    df["stock_code"] = df["stock_code"].astype("category")
    df["country"] = df["country"].astype("category")
    df["invoice"] = df["invoice"].astype("category")

    return df[["stock_code", "country", "date", "invoice", "quantity", "price", "revenue"]]


def aggregate_daily_from_chunks(path: Path) -> pd.DataFrame:
    daily_parts = []
    max_parts = 25
    for chunk in pd.read_csv(
        path,
        usecols=USECOLS,
        dtype=DTYPES,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        cleaned = preprocess_chunk(chunk)
        daily = cleaned.groupby(["stock_code", "country", "date"], as_index=False, observed=True).agg(
            demand_qty=("quantity", "sum"),
            revenue=("revenue", "sum"),
            num_invoices=("invoice", "nunique"),
            price_mean=("price", "mean"),
        )
        daily_parts.append(daily)
        if len(daily_parts) >= max_parts:
            partial = pd.concat(daily_parts, ignore_index=True)
            daily_parts = [
                partial.groupby(["stock_code", "country", "date"], as_index=False, observed=True)
                .agg(
                    demand_qty=("demand_qty", "sum"),
                    revenue=("revenue", "sum"),
                    num_invoices=("num_invoices", "sum"),
                    price_mean=("price_mean", "mean"),
                )
            ]
            del partial
            gc.collect()
        del chunk, cleaned, daily
        gc.collect()

    daily_all = pd.concat(daily_parts, ignore_index=True)
    del daily_parts
    gc.collect()
    daily_all = daily_all.groupby(["stock_code", "country", "date"], as_index=False, observed=True).agg(
        demand_qty=("demand_qty", "sum"),
        revenue=("revenue", "sum"),
        num_invoices=("num_invoices", "sum"),
        price_mean=("price_mean", "mean"),
    )

    daily_all["avg_price"] = np.where(
        daily_all["demand_qty"] > 0,
        daily_all["revenue"] / daily_all["demand_qty"],
        daily_all["price_mean"],
    )
    daily_all = daily_all.drop(columns=["price_mean"])

    daily_all["stock_code"] = daily_all["stock_code"].astype("category")
    daily_all["country"] = daily_all["country"].astype("category")

    daily_all["demand_qty"] = daily_all["demand_qty"].astype("float32")
    daily_all["revenue"] = daily_all["revenue"].astype("float32")
    daily_all["avg_price"] = daily_all["avg_price"].astype("float32")
    daily_all["num_invoices"] = daily_all["num_invoices"].astype("float32")

    return daily_all


daily_df = aggregate_daily_from_chunks(RAW_PATH)
daily_df = daily_df.sort_values(["stock_code", "country", "date"]).reset_index(drop=True)
daily_df.head()


## 3) Build full daily panel (to represent zero-sale days)

In [ ]:
def build_full_daily_panel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["stock_code", "country", "date"]).reset_index(drop=True)

    def _resample(group: pd.DataFrame) -> pd.DataFrame:
        group = group.set_index("date").asfreq("D")
        group["stock_code"] = group["stock_code"].iloc[0]
        group["country"] = group["country"].iloc[0]
        return group.reset_index()

    panel = df.groupby(["stock_code", "country"], group_keys=False, sort=False).apply(_resample)
    panel = panel.reset_index(drop=True)

    fill_cols = ["demand_qty", "revenue", "num_invoices"]
    panel[fill_cols] = panel[fill_cols].fillna(0)

    panel["avg_price"] = panel["avg_price"].astype("float32")
    panel["avg_price"] = panel.groupby(["stock_code", "country"], sort=False)["avg_price"].ffill().bfill().fillna(0)

    panel["stock_code"] = panel["stock_code"].astype("category")
    panel["country"] = panel["country"].astype("category")

    panel["demand_qty"] = panel["demand_qty"].astype("float32")
    panel["revenue"] = panel["revenue"].astype("float32")
    panel["avg_price"] = panel["avg_price"].astype("float32")
    panel["num_invoices"] = panel["num_invoices"].astype("float32")

    return panel


panel_df = build_full_daily_panel(daily_df)
del daily_df
gc.collect()
panel_df = panel_df.sort_values(["stock_code", "country", "date"]).reset_index(drop=True)
panel_df.head()


## 4) Calendar + holiday features

In [ ]:
import holidays

COUNTRY_TO_HOLIDAYS = {
    "Australia": "AU",
    "Austria": "AT",
    "Bahrain": "BH",
    "Belgium": "BE",
    "Bermuda": "BM",
    "Brazil": "BR",
    "Canada": "CA",
    "Channel Islands": "GB",
    "Cyprus": "CY",
    "Czech Republic": "CZ",
    "Denmark": "DK",
    "EIRE": "IE",
    "European Community": None,
    "Finland": "FI",
    "France": "FR",
    "Germany": "DE",
    "Greece": "GR",
    "Hong Kong": "HK",
    "Iceland": "IS",
    "Israel": "IL",
    "Italy": "IT",
    "Japan": "JP",
    "Korea": "KR",
    "Lebanon": "LB",
    "Lithuania": "LT",
    "Malta": "MT",
    "Netherlands": "NL",
    "Nigeria": "NG",
    "Norway": "NO",
    "Poland": "PL",
    "Portugal": "PT",
    "RSA": "ZA",
    "Saudi Arabia": "SA",
    "Singapore": "SG",
    "Spain": "ES",
    "Sweden": "SE",
    "Switzerland": "CH",
    "Thailand": "TH",
    "USA": "US",
    "United Arab Emirates": "AE",
    "United Kingdom": "GB",
    "Unspecified": None,
    "West Indies": None,
}

panel_df["country_code"] = panel_df["country"].map(COUNTRY_TO_HOLIDAYS).astype("category")

years = panel_df["date"].dt.year.unique().tolist()
holiday_rows = []
supported = set(holidays.list_supported_countries())
for code in sorted(panel_df["country_code"].dropna().astype(str).unique()):
    if code not in supported:
        continue
    holiday_set = holidays.country_holidays(code, years=years)
    holiday_rows.append(
        pd.DataFrame({
            "country_code": code,
            "date": list(holiday_set.keys()),
            "is_hari_besar": 1,
        })
    )

holiday_df = pd.concat(holiday_rows, ignore_index=True) if holiday_rows else pd.DataFrame(
    columns=["country_code", "date", "is_hari_besar"]
)
holiday_df["date"] = pd.to_datetime(holiday_df["date"], errors="coerce")

panel_df = panel_df.merge(holiday_df, on=["country_code", "date"], how="left", copy=False)
if "is_hari_besar" not in panel_df.columns:
    panel_df["is_hari_besar"] = 0
panel_df["is_hari_besar"] = panel_df["is_hari_besar"].fillna(0).astype("uint8")

pre_holiday_rows = []
if not holiday_df.empty:
    for offset in [1, 2, 3]:
        pre_holiday_rows.append(
            holiday_df.assign(
                date=holiday_df["date"] - pd.Timedelta(days=offset),
                is_pre_hari_besar=1,
            )[["country_code", "date", "is_pre_hari_besar"]]
        )

pre_holiday_df = pd.concat(pre_holiday_rows, ignore_index=True) if pre_holiday_rows else pd.DataFrame(
    columns=["country_code", "date", "is_pre_hari_besar"]
)
pre_holiday_df = pre_holiday_df.drop_duplicates()

panel_df = panel_df.merge(pre_holiday_df, on=["country_code", "date"], how="left", copy=False)
del pre_holiday_df, pre_holiday_rows, holiday_df, holiday_rows
gc.collect()
panel_df["is_pre_hari_besar"] = panel_df["is_pre_hari_besar"].fillna(0).astype("uint8")

panel_df["day_of_week"] = panel_df["date"].dt.dayofweek.astype("int8")
panel_df["week_of_year"] = panel_df["date"].dt.isocalendar().week.astype("int16")
panel_df["month"] = panel_df["date"].dt.month.astype("int8")
panel_df["quarter"] = panel_df["date"].dt.quarter.astype("int8")
panel_df["day_of_month"] = panel_df["date"].dt.day.astype("int8")
panel_df["is_weekend"] = (panel_df["day_of_week"] >= 5).astype("uint8")
panel_df["is_month_start"] = panel_df["date"].dt.is_month_start.astype("uint8")
panel_df["is_month_end"] = panel_df["date"].dt.is_month_end.astype("uint8")

month_end = panel_df["date"] + pd.offsets.MonthEnd(0)
panel_df["days_to_month_end"] = (month_end - panel_df["date"]).dt.days.astype("int16")

panel_df["week_of_month"] = ((panel_df["date"].dt.day - 1) // 7 + 1).astype("int8")
panel_df["is_month_start_window"] = (panel_df["date"].dt.day <= 5).astype("uint8")
panel_df["is_month_end_window"] = (panel_df["date"].dt.day >= 25).astype("uint8")

gc.collect()
panel_df.head()


## 5) Lag + rolling features (leakage-safe, shift=1)

In [ ]:
group_cols = ["stock_code", "country"]
panel_df = panel_df.sort_values(group_cols + ["date"]).reset_index(drop=True)

demand_shifted = panel_df.groupby(group_cols)["demand_qty"].shift(1)
panel_df["demand_lag_1"] = demand_shifted
panel_df["demand_lag_2"] = panel_df.groupby(group_cols)["demand_qty"].shift(2)
panel_df["demand_lag_7"] = panel_df.groupby(group_cols)["demand_qty"].shift(7)
panel_df["demand_lag_14"] = panel_df.groupby(group_cols)["demand_qty"].shift(14)
panel_df["demand_lag_21"] = panel_df.groupby(group_cols)["demand_qty"].shift(21)
panel_df["demand_lag_28"] = panel_df.groupby(group_cols)["demand_qty"].shift(28)
panel_df["demand_lag_35"] = panel_df.groupby(group_cols)["demand_qty"].shift(35)
panel_df["demand_lag_56"] = panel_df.groupby(group_cols)["demand_qty"].shift(56)
panel_df["demand_lag_84"] = panel_df.groupby(group_cols)["demand_qty"].shift(84)

panel_df["roll_max_7"] = demand_shifted.rolling(window=7, min_periods=1).max().values
panel_df["roll_max_28"] = demand_shifted.rolling(window=28, min_periods=1).max().values
panel_df["roll_zero_count_14"] = (demand_shifted == 0).rolling(window=14, min_periods=1).sum().values

panel_df["roll_mean_7"] = demand_shifted.rolling(window=7, min_periods=1).mean().values
panel_df["roll_mean_14"] = demand_shifted.rolling(window=14, min_periods=1).mean().values
panel_df["roll_mean_28"] = demand_shifted.rolling(window=28, min_periods=1).mean().values
panel_df["roll_mean_56"] = demand_shifted.rolling(window=56, min_periods=1).mean().values
panel_df["roll_median_7"] = demand_shifted.rolling(window=7, min_periods=1).median().values
panel_df["roll_median_14"] = demand_shifted.rolling(window=14, min_periods=1).median().values
panel_df["roll_median_28"] = demand_shifted.rolling(window=28, min_periods=1).median().values
panel_df["roll_std_7"] = demand_shifted.rolling(window=7, min_periods=1).std().values
panel_df["roll_std_14"] = demand_shifted.rolling(window=14, min_periods=1).std().values
panel_df["roll_std_28"] = demand_shifted.rolling(window=28, min_periods=1).std().values
panel_df["roll_std_56"] = demand_shifted.rolling(window=56, min_periods=1).std().values
panel_df["roll_max_56"] = demand_shifted.rolling(window=56, min_periods=1).max().values
panel_df["roll_max_84"] = demand_shifted.rolling(window=84, min_periods=1).max().values

roll_mean_3 = demand_shifted.rolling(window=3, min_periods=1).mean().values
roll_mean_14 = demand_shifted.rolling(window=14, min_periods=1).mean().values
panel_df["demand_acceleration_3d"] = roll_mean_3 / (roll_mean_14 + 1e-8)
panel_df["spike_ratio_28"] = panel_df["roll_max_28"] / (panel_df["roll_mean_28"] + 1e-8)
panel_df["spike_ratio_56"] = panel_df["roll_max_56"] / (panel_df["roll_mean_56"] + 1e-8)

panel_df["pct_change_1"] = (panel_df["demand_lag_1"] - panel_df["demand_lag_2"]) / (panel_df["demand_lag_2"] + 1e-8)
panel_df["pct_change_7"] = (panel_df["demand_lag_7"] - panel_df["demand_lag_14"]) / (panel_df["demand_lag_14"] + 1e-8)

last_sale_date = panel_df["date"].where(demand_shifted > 0)
last_sale_date = last_sale_date.groupby(panel_df[group_cols].apply(tuple, axis=1)).ffill()
panel_df["days_since_last_sale"] = (panel_df["date"] - last_sale_date).dt.days
panel_df["days_since_last_sale"] = panel_df["days_since_last_sale"].fillna(9999).astype("int16")

price_shifted = panel_df.groupby(group_cols)["avg_price"].shift(1)
roll_price_max_30 = price_shifted.rolling(window=30, min_periods=1).max().values
panel_df["discount_depth_pct"] = (roll_price_max_30 - panel_df["avg_price"]) / (roll_price_max_30 + 1e-8)

roll_price_mean_14 = price_shifted.rolling(window=14, min_periods=1).mean().values
panel_df["price_momentum"] = panel_df["avg_price"] / (roll_price_mean_14 + 1e-8)

numeric_cols = panel_df.select_dtypes(include=["number"]).columns
panel_df[numeric_cols] = panel_df[numeric_cols].replace([np.inf, -np.inf], 0).fillna(0)
for col in panel_df.select_dtypes(include=["float64"]).columns:
    panel_df[col] = panel_df[col].astype("float32")
gc.collect()
panel_df.head()


## 6) Time Series CV setup

In [ ]:
FEATURE_COLS = [
    "day_of_week", "week_of_year", "month", "quarter", "day_of_month",
    "is_weekend", "is_month_start", "is_month_end",
    "days_to_month_end", "week_of_month", "is_month_start_window", "is_month_end_window",
    "is_hari_besar", "is_pre_hari_besar",
    "demand_lag_1", "demand_lag_2", "demand_lag_7", "demand_lag_14", "demand_lag_21", "demand_lag_28", "demand_lag_35", "demand_lag_56", "demand_lag_84",
    "days_since_last_sale", "roll_zero_count_14",
    "roll_max_7", "roll_max_28",
    "roll_mean_7", "roll_mean_14", "roll_mean_28", "roll_mean_56",
    "roll_median_7", "roll_median_14", "roll_median_28",
    "roll_std_7", "roll_std_14", "roll_std_28", "roll_std_56",
    "roll_max_56", "roll_max_84",
    "demand_acceleration_3d",
    "spike_ratio_28", "spike_ratio_56",
    "pct_change_1", "pct_change_7",
    "discount_depth_pct",
    "price_momentum",
]

TARGET_COL = "demand_qty"
PRICE_COL = "avg_price"
DATE_COL = "date"

def time_series_folds(dates, horizon_days=30, n_splits=3, min_train_days=180):
    dates = np.array(sorted(pd.to_datetime(dates).unique()))
    total_days = len(dates)
    splits = []
    for i in range(n_splits):
        val_end_idx = total_days - (n_splits - i - 1) * horizon_days
        val_start_idx = val_end_idx - horizon_days
        train_end_idx = val_start_idx - 1
        if train_end_idx < min_train_days:
            continue
        train_end = dates[train_end_idx]
        val_start = dates[val_start_idx]
        val_end = dates[val_end_idx - 1]
        splits.append((train_end, val_start, val_end))
    return splits

splits = time_series_folds(panel_df[DATE_COL], horizon_days=30, n_splits=3, min_train_days=180)
splits


## 7) Metrics and asymmetric loss

In [ ]:
from sklearn.metrics import mean_absolute_error, f1_score, precision_score, recall_score
from sklearn.calibration import CalibratedClassifierCV

def calc_cls(y_true, y_pred, avg_price_arr, margin=0.20):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    return float(np.sum(np.maximum(y_true - y_pred, 0) * avg_price * margin))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    if mask.sum() == 0:
        return 0.0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def evaluate_prediction(y_true, y_pred, avg_price_arr, baseline_mae=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))
    smape_val = smape(y_true, y_pred)
    y_true_bin = (np.asarray(y_true) > 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) > 0).astype(int)
    f1_zero = f1_score(y_true_bin, y_pred_bin)
    precision_zero = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall_zero = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    cls = calc_cls(y_true, y_pred, avg_price_arr)
    ofr = np.minimum(y_true, y_pred).sum() / (y_true.sum() + 1e-8)
    oos_rate = float(np.mean(y_pred < y_true))
    fva = None
    if baseline_mae is not None and baseline_mae > 0:
        fva = float((baseline_mae - mae) / baseline_mae)
    return {"mae": mae, "rmse": rmse, "smape": smape_val, "f1_zero": f1_zero,
            "precision_zero": precision_zero, "recall_zero": recall_zero,
            "cls": cls, "ofr": ofr, "oos_rate": oos_rate, "fva": fva}

def asymmetric_obj(alpha=2.0):
    def obj(y_true, y_pred, sample_weight=None):
        residual = y_pred - y_true
        grad = np.where(residual > 0, 2 * residual, 2 * alpha * residual)
        hess = np.where(residual > 0, 2.0, 2.0 * alpha)
        if sample_weight is not None:
            grad = grad * sample_weight
            hess = hess * sample_weight
        return grad, hess
    return obj

def quantile_obj(q=0.8):
    def obj(y_true, y_pred, sample_weight=None):
        residual = y_pred - y_true
        grad = np.where(residual >= 0, q, q - 1.0)
        hess = np.ones_like(grad) * 1e-6
        if sample_weight is not None:
            grad = grad * sample_weight
            hess = hess * sample_weight
        return grad, hess
    return obj


## 8) Twin-XGB training with TS CV

In [ ]:
CLF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "max_bin": 256,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

REG_PARAMS = {
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "max_bin": 256,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

ALPHA_UNDER = 50.0
QUANTILE_Q = 0.8
QUANTILE_Q_TOP = 0.9
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.05), 2).tolist()
USE_LOG_TARGET = True
TOP_SEGMENT_PCT = 0.05
SAMPLE_WEIGHT_ALPHA = 4.0
SAMPLE_WEIGHT_CAP = 5.0

fold_metrics = []

for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    train_mask = panel_df[DATE_COL] <= train_end
    val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

    train_segment = (
        panel_df.loc[train_mask]
        .groupby(["stock_code", "country"])[TARGET_COL]
        .sum()
        .sort_values(ascending=False)
    )
    top_n = max(1, int(len(train_segment) * TOP_SEGMENT_PCT))
    top_keys = set(train_segment.head(top_n).index)

    df_train = panel_df.loc[train_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, "stock_code", "country"]]
    df_val = panel_df.loc[val_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, "stock_code", "country"]]

    X_train = df_train[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    X_val = df_val[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_val = df_val[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    price_val = df_val[PRICE_COL].to_numpy(dtype=np.float32, copy=False)
    train_keys = list(zip(df_train["stock_code"].to_numpy(), df_train["country"].to_numpy()))
    val_keys = list(zip(df_val["stock_code"].to_numpy(), df_val["country"].to_numpy()))
    train_is_top = np.array([key in top_keys for key in train_keys])
    val_is_top = np.array([key in top_keys for key in val_keys])

    y_train_zero = (y_train > 0).astype(int)
    y_val_zero = (y_val > 0).astype(int)

    scale_pos_weight = (len(y_train_zero) - y_train_zero.sum()) / (y_train_zero.sum() + 1e-8)
    clf = xgb.XGBClassifier(**CLF_PARAMS, objective="binary:logistic", scale_pos_weight=scale_pos_weight)
    clf.fit(X_train, y_train_zero)
    calibrator = CalibratedClassifierCV(clf, method="isotonic", cv=3)
    calibrator.fit(X_train, y_train_zero)

    nonzero_mask = y_train > 0
    y_train_nonzero = y_train[nonzero_mask]
    y_train_reg = np.log1p(y_train_nonzero) if USE_LOG_TARGET else y_train_nonzero
    q95 = np.quantile(y_train_nonzero, 0.95) if y_train_nonzero.size else 0.0
    weight_scale = (y_train_nonzero / (q95 + 1e-8)) if q95 > 0 else np.zeros_like(y_train_nonzero)
    sample_weight = 1.0 + SAMPLE_WEIGHT_ALPHA * np.minimum(weight_scale, SAMPLE_WEIGHT_CAP)

    reg_top = xgb.XGBRegressor(**REG_PARAMS, objective="reg:quantileerror", quantile_alpha=QUANTILE_Q_TOP)
    reg_tail = xgb.XGBRegressor(**REG_PARAMS, objective=asymmetric_obj(ALPHA_UNDER))

    reg_top.fit(X_train[nonzero_mask], y_train_reg, sample_weight=sample_weight)
    reg_tail.fit(X_train[nonzero_mask], y_train_reg)

    prob_nonzero = calibrator.predict_proba(X_val)[:, 1]
    pred_reg_top = reg_top.predict(X_val)
    pred_reg_tail = reg_tail.predict(X_val)
    if USE_LOG_TARGET:
        pred_reg_top = np.expm1(pred_reg_top)
        pred_reg_tail = np.expm1(pred_reg_tail)
    pred_reg_top = np.maximum(pred_reg_top, 0)
    pred_reg_tail = np.maximum(pred_reg_tail, 0)
    pred_reg = np.where(val_is_top, pred_reg_top, pred_reg_tail)
    if val_is_top.any():
        residual_top = pred_reg_top[val_is_top] - y_val[val_is_top]
        q95 = np.quantile(y_val[val_is_top], 0.95)
        tail_mask = y_val[val_is_top] >= q95
        tail_residual = residual_top[tail_mask] if tail_mask.any() else residual_top
        tail_mean = np.mean(y_val[val_is_top][tail_mask]) if tail_mask.any() else np.mean(y_val[val_is_top])
        top_beta = max(0.0, -np.mean(tail_residual) / (tail_mean + 1e-8))
        pred_reg = np.where(val_is_top, pred_reg * (1.0 + top_beta), pred_reg)

    baseline_mae = float(np.mean(np.abs(y_val)))
    best_threshold = None
    best_cls = None
    best_metrics = None

    for threshold in THRESHOLD_GRID:
        pred_zero = (prob_nonzero >= threshold).astype(int)
        pred = pred_reg * pred_zero
        metrics = evaluate_prediction(y_val, pred, price_val, baseline_mae=baseline_mae)
        if best_cls is None or metrics["cls"] < best_cls:
            best_cls = metrics["cls"]
            best_threshold = threshold
            best_metrics = metrics

    best_metrics["threshold"] = best_threshold
    best_metrics["fold"] = fold_idx
    best_metrics["train_end"] = str(train_end.date())
    best_metrics["val_start"] = str(val_start.date())
    best_metrics["val_end"] = str(val_end.date())
    best_metrics["use_log_target"] = USE_LOG_TARGET
    best_metrics["calibration"] = "isotonic"
    best_metrics["quantile_q"] = QUANTILE_Q
    best_metrics["top_segment_pct"] = TOP_SEGMENT_PCT
    best_metrics["sample_weight_alpha"] = SAMPLE_WEIGHT_ALPHA
    best_metrics["sample_weight_cap"] = SAMPLE_WEIGHT_CAP
    fold_metrics.append(best_metrics)

    del df_train, df_val, X_train, y_train, X_val, y_val, price_val, clf, reg_top, reg_tail, calibrator, pred_reg
    gc.collect()

pd.DataFrame(fold_metrics)


## 9) Summary

In [ ]:
summary_df = pd.DataFrame(fold_metrics)
summary_df


## 9.1) Segment evaluation (top-demand vs long-tail)

In [ ]:
TOP_SEGMENT_PCT = 0.05

segment_rows = []
for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    train_mask = panel_df[DATE_COL] <= train_end
    val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

    train_segment = (
        panel_df.loc[train_mask]
        .groupby(["stock_code", "country"])
        [TARGET_COL].sum()
        .sort_values(ascending=False)
    )
    top_n = max(1, int(len(train_segment) * TOP_SEGMENT_PCT))
    top_keys = set(train_segment.head(top_n).index)

    val_df = panel_df.loc[val_mask, ["stock_code", "country", TARGET_COL, PRICE_COL]].copy()
    val_df["key"] = list(zip(val_df["stock_code"], val_df["country"]))
    val_df["segment"] = np.where(val_df["key"].isin(top_keys), "top", "tail")

    best_threshold = summary_df.loc[summary_df["fold"] == fold_idx, "threshold"].iloc[0]

    df_train = panel_df.loc[train_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, "stock_code", "country"]]
    df_val = panel_df.loc[val_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, "stock_code", "country"]]

    X_train = df_train[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    X_val = df_val[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_val = df_val[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    price_val = df_val[PRICE_COL].to_numpy(dtype=np.float32, copy=False)
    train_keys = list(zip(df_train["stock_code"].to_numpy(), df_train["country"].to_numpy()))
    val_keys = list(zip(df_val["stock_code"].to_numpy(), df_val["country"].to_numpy()))
    train_is_top = np.array([key in top_keys for key in train_keys])
    val_is_top = np.array([key in top_keys for key in val_keys])

    y_train_zero = (y_train > 0).astype(int)
    scale_pos_weight = (len(y_train_zero) - y_train_zero.sum()) / (y_train_zero.sum() + 1e-8)
    clf = xgb.XGBClassifier(**CLF_PARAMS, objective="binary:logistic", scale_pos_weight=scale_pos_weight)
    clf.fit(X_train, y_train_zero)
    calibrator = CalibratedClassifierCV(clf, method="isotonic", cv=3)
    calibrator.fit(X_train, y_train_zero)

    nonzero_mask = y_train > 0
    y_train_nonzero = y_train[nonzero_mask]
    y_train_reg = np.log1p(y_train_nonzero) if USE_LOG_TARGET else y_train_nonzero
    q95 = np.quantile(y_train_nonzero, 0.95) if y_train_nonzero.size else 0.0
    weight_scale = (y_train_nonzero / (q95 + 1e-8)) if q95 > 0 else np.zeros_like(y_train_nonzero)
    sample_weight = 1.0 + SAMPLE_WEIGHT_ALPHA * np.minimum(weight_scale, SAMPLE_WEIGHT_CAP)

    reg_top = xgb.XGBRegressor(**REG_PARAMS, objective="reg:quantileerror", quantile_alpha=QUANTILE_Q_TOP)
    reg_tail = xgb.XGBRegressor(**REG_PARAMS, objective=asymmetric_obj(ALPHA_UNDER))

    reg_top.fit(X_train[nonzero_mask], y_train_reg, sample_weight=sample_weight)
    reg_tail.fit(X_train[nonzero_mask], y_train_reg)

    prob_nonzero = calibrator.predict_proba(X_val)[:, 1]
    pred_reg_top = reg_top.predict(X_val)
    pred_reg_tail = reg_tail.predict(X_val)
    if USE_LOG_TARGET:
        pred_reg_top = np.expm1(pred_reg_top)
        pred_reg_tail = np.expm1(pred_reg_tail)
    pred_reg_top = np.maximum(pred_reg_top, 0)
    pred_reg_tail = np.maximum(pred_reg_tail, 0)
    pred_reg = np.where(val_is_top, pred_reg_top, pred_reg_tail)
    if val_is_top.any():
        residual_top = pred_reg_top[val_is_top] - y_val[val_is_top]
        q95 = np.quantile(y_val[val_is_top], 0.95)
        tail_mask = y_val[val_is_top] >= q95
        tail_residual = residual_top[tail_mask] if tail_mask.any() else residual_top
        tail_mean = np.mean(y_val[val_is_top][tail_mask]) if tail_mask.any() else np.mean(y_val[val_is_top])
        top_beta = max(0.0, -np.mean(tail_residual) / (tail_mean + 1e-8))
        pred_reg = np.where(val_is_top, pred_reg * (1.0 + top_beta), pred_reg)

    pred_zero = (prob_nonzero >= best_threshold).astype(int)
    pred = pred_reg * pred_zero

    val_df["forecast"] = pred

    for segment_name, seg in val_df.groupby("segment"):
        metrics = evaluate_prediction(seg[TARGET_COL], seg["forecast"], seg[PRICE_COL])
        metrics["fold"] = fold_idx
        metrics["segment"] = segment_name
        metrics["threshold"] = best_threshold
        segment_rows.append(metrics)

    del df_train, df_val, X_train, y_train, X_val, y_val, price_val, clf, reg_top, reg_tail, calibrator, pred_reg
    gc.collect()

segment_df = pd.DataFrame(segment_rows)
segment_df

segment_summary = segment_df.groupby("segment").agg(
    mae=("mae", "mean"),
    rmse=("rmse", "mean"),
    smape=("smape", "mean"),
    cls=("cls", "mean"),
    ofr=("ofr", "mean"),
    oos_rate=("oos_rate", "mean"),
).reset_index()
segment_summary


## 10) Forecast performance visualization (top N total demand)

In [ ]:
import matplotlib.pyplot as plt

TOP_N = 5

top_items = (
    panel_df.groupby(["stock_code", "country"])["demand_qty"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N)
    .reset_index()
)
top_keys = {tuple(row) for row in top_items[["stock_code", "country"]].to_numpy()}

pred_rows = []
for fold_idx, (train_end, val_start, val_end) in enumerate(splits, start=1):
    train_mask = panel_df[DATE_COL] <= train_end
    val_mask = (panel_df[DATE_COL] >= val_start) & (panel_df[DATE_COL] <= val_end)

    df_train = panel_df.loc[train_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, "stock_code", "country"]]
    df_val = panel_df.loc[val_mask, FEATURE_COLS + [TARGET_COL, PRICE_COL, DATE_COL, "stock_code", "country"]]

    X_train = df_train[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    X_val = df_val[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
    y_val = df_val[TARGET_COL].to_numpy(dtype=np.float32, copy=False)
    price_val = df_val[PRICE_COL].to_numpy(dtype=np.float32, copy=False)
    train_keys = list(zip(df_train["stock_code"].to_numpy(), df_train["country"].to_numpy()))
    val_keys = list(zip(df_val["stock_code"].to_numpy(), df_val["country"].to_numpy()))
    train_is_top = np.array([key in top_keys for key in train_keys])
    val_is_top = np.array([key in top_keys for key in val_keys])

    y_train_zero = (y_train > 0).astype(int)
    scale_pos_weight = (len(y_train_zero) - y_train_zero.sum()) / (y_train_zero.sum() + 1e-8)
    clf = xgb.XGBClassifier(**CLF_PARAMS, objective="binary:logistic", scale_pos_weight=scale_pos_weight)
    clf.fit(X_train, y_train_zero)
    calibrator = CalibratedClassifierCV(clf, method="isotonic", cv=3)
    calibrator.fit(X_train, y_train_zero)

    nonzero_mask = y_train > 0
    y_train_nonzero = y_train[nonzero_mask]
    y_train_reg = np.log1p(y_train_nonzero) if USE_LOG_TARGET else y_train_nonzero
    q95 = np.quantile(y_train_nonzero, 0.95) if y_train_nonzero.size else 0.0
    weight_scale = (y_train_nonzero / (q95 + 1e-8)) if q95 > 0 else np.zeros_like(y_train_nonzero)
    sample_weight = 1.0 + SAMPLE_WEIGHT_ALPHA * np.minimum(weight_scale, SAMPLE_WEIGHT_CAP)

    reg_top = xgb.XGBRegressor(**REG_PARAMS, objective="reg:quantileerror", quantile_alpha=QUANTILE_Q_TOP)
    reg_tail = xgb.XGBRegressor(**REG_PARAMS, objective=asymmetric_obj(ALPHA_UNDER))

    reg_top.fit(X_train[nonzero_mask], y_train_reg, sample_weight=sample_weight)
    reg_tail.fit(X_train[nonzero_mask], y_train_reg)

    prob_nonzero = calibrator.predict_proba(X_val)[:, 1]
    pred_reg_top = reg_top.predict(X_val)
    pred_reg_tail = reg_tail.predict(X_val)
    if USE_LOG_TARGET:
        pred_reg_top = np.expm1(pred_reg_top)
        pred_reg_tail = np.expm1(pred_reg_tail)
    pred_reg_top = np.maximum(pred_reg_top, 0)
    pred_reg_tail = np.maximum(pred_reg_tail, 0)
    pred_reg = np.where(val_is_top, pred_reg_top, pred_reg_tail)
    if val_is_top.any():
        residual_top = pred_reg_top[val_is_top] - y_val[val_is_top]
        q95 = np.quantile(y_val[val_is_top], 0.95)
        tail_mask = y_val[val_is_top] >= q95
        tail_residual = residual_top[tail_mask] if tail_mask.any() else residual_top
        tail_mean = np.mean(y_val[val_is_top][tail_mask]) if tail_mask.any() else np.mean(y_val[val_is_top])
        top_beta = max(0.0, -np.mean(tail_residual) / (tail_mean + 1e-8))
        pred_reg = np.where(val_is_top, pred_reg * (1.0 + top_beta), pred_reg)

    best_threshold = None
    best_cls = None
    for threshold in THRESHOLD_GRID:
        pred_zero = (prob_nonzero >= threshold).astype(int)
        pred = pred_reg * pred_zero
        cls_val = calc_cls(y_val, pred, price_val)
        if best_cls is None or cls_val < best_cls:
            best_cls = cls_val
            best_threshold = threshold

    pred_zero = (prob_nonzero >= best_threshold).astype(int)
    pred = pred_reg * pred_zero

    fold_pred = df_val[[DATE_COL, "stock_code", "country", PRICE_COL]].copy()
    fold_pred["actual"] = y_val
    fold_pred["forecast"] = pred
    fold_pred["fold"] = fold_idx
    fold_pred["threshold"] = best_threshold
    pred_rows.append(fold_pred)

    del df_train, df_val, X_train, y_train, X_val, y_val, price_val, clf, reg_top, reg_tail, calibrator, pred_reg
    gc.collect()

pred_df = pd.concat(pred_rows, ignore_index=True)
pred_df["key"] = list(zip(pred_df["stock_code"], pred_df["country"]))
top_pred_df = pred_df[pred_df["key"].isin(top_keys)].copy()

metric_rows = []
for (stock_code, country), group in top_pred_df.groupby(["stock_code", "country"]):
    metrics = evaluate_prediction(group["actual"], group["forecast"], group[PRICE_COL])
    metrics["stock_code"] = stock_code
    metrics["country"] = country
    metric_rows.append(metrics)

top_metrics_df = pd.DataFrame(metric_rows).sort_values("mae")
top_metrics_df

for (stock_code, country), group in top_pred_df.groupby(["stock_code", "country"]):
    group = group.sort_values(DATE_COL)
    total_demand = top_items.loc[(top_items["stock_code"] == stock_code) & (top_items["country"] == country), "demand_qty"].iloc[0]
    plt.figure(figsize=(12, 4))
    plt.plot(group[DATE_COL], group["actual"], label="actual", linewidth=2)
    plt.plot(group[DATE_COL], group["forecast"], label="forecast", linewidth=2)
    plt.title(f"stock_code={stock_code} | country={country} | total_demand={total_demand:.0f}")
    plt.xlabel("date")
    plt.ylabel("demand_qty")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 11) Residual and peak diagnostics (top demand)

In [ ]:
focus_fold = max(pred_df["fold"].unique())
focus_df = top_pred_df[top_pred_df["fold"] == focus_fold].copy()

for (stock_code, country), group in focus_df.groupby(["stock_code", "country"]):
    group = group.sort_values(DATE_COL)
    residual = group["forecast"] - group["actual"]

    plt.figure(figsize=(12, 3))
    plt.plot(group[DATE_COL], residual, label="residual", color="tab:orange")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"Residuals | stock_code={stock_code} | country={country} | fold={focus_fold}")
    plt.xlabel("date")
    plt.ylabel("forecast - actual")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(5, 5))
    plt.scatter(group["actual"], group["forecast"], alpha=0.5)
    max_val = max(group["actual"].max(), group["forecast"].max())
    plt.plot([0, max_val], [0, max_val], color="black", linewidth=1)
    plt.title(f"Actual vs Forecast | stock_code={stock_code} | country={country}")
    plt.xlabel("actual")
    plt.ylabel("forecast")
    plt.tight_layout()
    plt.show()


## 12) Error by demand quantile (non-zero)

In [ ]:
quantile_rows = []
for fold_idx in sorted(pred_df["fold"].unique()):
    fold_df = pred_df[pred_df["fold"] == fold_idx].copy()
    fold_df = fold_df[fold_df["actual"] > 0].copy()
    if fold_df.empty:
        continue
    fold_df["q_bucket"] = pd.qcut(
        fold_df["actual"],
        q=[0.0, 0.5, 0.8, 0.95, 1.0],
        labels=["0-50", "50-80", "80-95", "95-100"],
        duplicates="drop",
    )
    for bucket, group in fold_df.groupby("q_bucket"):
        metrics = evaluate_prediction(group["actual"], group["forecast"], group[PRICE_COL])
        metrics["fold"] = fold_idx
        metrics["bucket"] = bucket
        quantile_rows.append(metrics)

quantile_df = pd.DataFrame(quantile_rows)
quantile_df
